## Drug discovery KG learning using TarKG

In [ ]:
!rm -rf drug-target-interaction-gnns
!git clone https://github.com/mylonasc/drug-target-interaction-gnns.git
!cp -r drug-target-interaction-gnns/dataset/tar_kg .
!cp -r drug-target-interaction-gnns/dataset/utils dataset_utils

In [ ]:
from dataset_utils.google_drive_cache import GoogleDriveCache
from tar_kg.tarkg_loader import TarKGLoader

In [ ]:
import shutil
import subprocess as sp
from google.colab import drive
from pathlib import Path



In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:

#@title Cache or load to/from google drive:

CACHE_TO_DRIVE = False #@param
DOWNLOAD_TARKG = False #@param
LOAD_FROM_DRIVE_CACHE = True #@param

cache = GoogleDriveCache(
    drive_cache_dir="/content/drive/MyDrive/datasets/DrugDiscovery/tar_kg_cache",
    colab_load_dir="/root/.datasets/tarkg_data/parquet",
    compress=False,
    verify_on_load=True,
)
if LOAD_FROM_DRIVE_CACHE:
  cache.untar_and_load(drive_path='/content/drive/MyDrive/datasets/DrugDiscovery/tar_kg_cache/parquet.tar', strip_top_level_dir=True)

In [ ]:
loader = TarKGLoader()

data = loader.load(
    download = DOWNLOAD_TARKG,
    use_parquet_cache = True,
    kg = True,
    lazy=True
)

# only useful the first time the dataset is loaded (subsequent loads will use the cached parquet files)
if CACHE_TO_DRIVE:
    cache.tar_and_cache_to_drive('/root/.datasets/tarkg_data/parquet/')

In [ ]:
!pip install -U "gestaltdb[fast-ingest]>=0.5.1" pyrex-rocksdb

In [ ]:
# !pip install pyrex-rocksdb
# !pip install --pre "pygraphdb[fast-ingest]"

In [ ]:
import gestaltdb as pg

In [ ]:
from gestaltdb import IndexMaintenanceMode
from gestaltdb.graphdb import GraphDB
from gestaltdb.serializers import JSONSerializer
from gestaltdb.kvstores import PyRexStore
if 'graphdb' in locals():
  graphdb.close()
  del graphdb
!rm -rf /tmp/targ_v3/
# graphdb = GraphDB(PyRexStore(path = '/tmp/targ_v3'), ProtobufSerializer())
graphdb = GraphDB(
    PyRexStore(
        path='/tmp/targ_v3',
        disable_wal=True,
        write_buffer_size=64 * 1024 * 1024,
        parallelism=4,
        max_background_jobs=4,
    ),
    JSONSerializer(),
)

In [ ]:
print('gestaltdb', getattr(pg, '__version__', 'unknown'))
print('store', type(graphdb.store).__name__)
print('native columnar ingestion', graphdb.store.has_native_columnar_ingestion())

In [ ]:
import polars as pl

nodes_raw = data['TarKG_nodes']
edges_raw = data['TarKG_edges']

node_df = nodes_raw.select(
    pl.col('unify_id').alias('node_id'),
    pl.concat_list('kind').alias('labels'),
    pl.col('kind'),
    pl.col('db_source'),
)
node_ids = node_df.select('node_id').unique()

In [ ]:
nodes_raw
  .select('db_source', 'source')
  .filter(pl.col('db_source').is_not_null())
  .group_by('source')
  .len()
  .sort('len', descending=True)
  .collect()

In [ ]:
# Preview a small node sample without materializing the full dataset.
node_df.head(5).collect()

In [ ]:
node_ingest_result = graphdb.ingest_nodes_polars_entities(
    node_df,
    node_id='node_id',
    labels='labels',
    property_columns=['kind', 'db_source'],
    index_mode=IndexMaintenanceMode.DEFER,
    chunk_size=100_000,
    progress=True,
)
node_ingest_result

In [ ]:
data['TarKG_edges'].filter(pl.col('node1') == 'TC4').head().collect()

In [ ]:
# Nodes were ingested lazily above; no Python Node list is built.

In [ ]:
# Keep node IDs as a LazyFrame so edge filtering does not require a large Python set.
node_ids

In [ ]:
edges_raw.head(n=1000).collect()

In [ ]:
# data['TarKG_edges'].select('index').collect()

In [ ]:
import time

FILTER_EDGES_TO_INGESTED_NODES = True
MATERIALIZE_FILTERED_EDGES = True
EDGES_PROC_PATH = '/tmp/tarkg_edges_proc.parquet'

edges_proc_lazy = (
    edges_raw
    .select([
        pl.col('index').cast(pl.Utf8).alias('edge_id'),
        pl.col('node1').alias('source'),
        pl.col('node2').alias('target'),
        pl.col('relation').alias('edge_type'),
        pl.col('node1_type'),
        pl.col('node2_type'),
    ])
)

if FILTER_EDGES_TO_INGESTED_NODES:
    edges_proc_lazy = (
        edges_proc_lazy
        .join(node_ids, left_on='source', right_on='node_id', how='semi')
        .join(node_ids, left_on='target', right_on='node_id', how='semi')
    )

if MATERIALIZE_FILTERED_EDGES:
    start = time.perf_counter()
    Path(EDGES_PROC_PATH).unlink(missing_ok=True)
    edges_proc_lazy.sink_parquet(EDGES_PROC_PATH)
    print(f'materialized filtered edges in {time.perf_counter() - start:.1f}s')
    edges_proc = pl.scan_parquet(EDGES_PROC_PATH)
    edge_count = edges_proc.select(pl.len()).collect().item()
    print(f'filtered edges: {edge_count:,}')
else:
    edges_proc = edges_proc_lazy

In [ ]:
# Preview the lazy edge frame after endpoint filtering.
edges_proc.head(5).collect()

In [ ]:
start = time.perf_counter()
edge_ingest_result = graphdb.ingest_edges_polars_entities(
    edges_proc,
    edge_id='edge_id',
    source='source',
    target='target',
    edge_type='edge_type',
    property_columns=['node1_type', 'node2_type'],
    index_mode=IndexMaintenanceMode.DEFER,
    chunk_size=100_000,
    progress=True,
)
edge_ingest_seconds = time.perf_counter() - start
print(f'ingested {edge_ingest_result:,} edges in {edge_ingest_seconds:.1f}s ({edge_ingest_result / edge_ingest_seconds:,.0f} edges/s)')
edge_ingest_result

In [ ]:
graphdb

In [ ]:

?graphdb.ingest_edges_polars_entities

In [ ]:
edges_proc

In [ ]:
edges_proc

In [ ]:
# Obsolete eager Edge-object ingestion removed.
# Use edge_ingest_result from the LazyFrame ingestion cell above.
edge_ingest_result

In [ ]:
!tar -czf /tmp/targ_v3.tar.gz /tmp/targ_v3 && cp /tmp/targ_v3.tar.gz /content/drive/MyDrive/datasets/DrugDiscovery/

In [ ]:
# No eager Python edge-building loop is needed.
edges_proc.head(5).collect()

In [ ]:
# Inspect the database wrapper after lazy ingestion.
graphdb

In [ ]:
# Edges were ingested lazily above; no Python Edge list is built.

In [ ]:
edge_ingest_result

In [ ]:
graphdb

In [ ]:
# Lazy source-frequency summary for exploration only.
nodes_raw.group_by('source').len().sort('len', descending=True).head(25).collect()

In [ ]:
# plyvel is installed through gestaltdb[fast-ingest] when needed.

In [ ]:
cache.untar_and_load(drive_path='/content/drive/MyDrive/datasets/DrugDiscovery/tar_kg_cache/parquet.tar', strip_top_level_dir=True)